# Instruction Tuning con SFTTrainer

> Aprende a preparar datasets de instrucciones en múltiples formatos y a configurar
> SFTTrainer con LoRA para fine-tuning eficiente en memoria.

## Objetivos
- Comprender los formatos Alpaca, ShareGPT y ChatML
- Generar un dataset sintético de instrucciones con Claude
- Configurar SFTTrainer con LoRA para instrucción tuning
- Entender los chat templates de diferentes modelos

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic trl transformers datasets peft torch --quiet

## 2. Formatos de datasets de instrucciones

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

# Formato Alpaca (el más simple y ampliamente usado)
ejemplo_alpaca = {
    "instruction": "Explica qué es el gradient descent en machine learning.",
    "input": "",  # Campo vacío si no hay contexto adicional
    "output": "El gradient descent es un algoritmo de optimización que ajusta los parámetros de un modelo minimizando una función de pérdida. En cada iteración, calcula el gradiente (derivada parcial) de la pérdida respecto a cada parámetro y los actualiza en la dirección opuesta al gradiente, avanzando hacia el mínimo."
}

# Formato ShareGPT (conversaciones multi-turno)
ejemplo_sharegpt = {
    "conversations": [
        {"from": "human", "value": "¿Qué es el overfitting?"},
        {"from": "gpt", "value": "El overfitting ocurre cuando un modelo aprende demasiado bien los datos de entrenamiento, incluyendo el ruido, y generaliza mal a datos nuevos."},
        {"from": "human", "value": "¿Cómo se detecta?"},
        {"from": "gpt", "value": "Se detecta comparando el error en entrenamiento vs validación: si el error de entrenamiento baja pero el de validación sube, hay overfitting."}
    ]
}

# Formato ChatML (OpenAI, usado por Mistral y otros)
ejemplo_chatml = """<|im_start|>system
Eres un asistente experto en machine learning.
<|im_end|>
<|im_start|>user
¿Qué es el regularization?
<|im_end|>
<|im_start|>assistant
La regularización es una técnica para prevenir el overfitting añadiendo un término de penalización a la función de pérdida. Las más comunes son L1 (Lasso) y L2 (Ridge).
<|im_end|>"""

print("=== FORMATOS DE DATASETS DE INSTRUCCIONES ===")
print("\n1. Formato Alpaca:")
print(json.dumps(ejemplo_alpaca, indent=2, ensure_ascii=False))
print("\n2. Formato ShareGPT (multi-turno):")
print(json.dumps(ejemplo_sharegpt, indent=2, ensure_ascii=False))
print("\n3. Formato ChatML:")
print(ejemplo_chatml)

## 3. Generar dataset sintético de instrucciones con Claude

In [ ]:
TEMAS = [
    "machine learning", "redes neuronales", "procesamiento de lenguaje natural",
    "visión por computador", "evaluación de modelos"
]

def generar_ejemplos_alpaca(tema: str, n: int = 3) -> list[dict]:
    """Claude genera ejemplos de instrucción-respuesta en formato Alpaca."""
    prompt = f"""Genera {n} ejemplos de instrucción-respuesta en formato Alpaca sobre el tema '{tema}'.
Cada ejemplo debe ser educativo, preciso y en español.

Devuelve SOLO una lista JSON:
[
  {{"instruction": "<pregunta o tarea>", "input": "", "output": "<respuesta detallada>"}},
  ...
]"""

    resp = client.messages.create(
        model=MODELO, max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()

    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json").strip()
    return json.loads(resp)

print("Generando dataset sintético con Claude...")
dataset_alpaca = []
for tema in TEMAS:
    ejemplos = generar_ejemplos_alpaca(tema, n=3)
    dataset_alpaca.extend(ejemplos)
    print(f"  ✓ {tema}: {len(ejemplos)} ejemplos generados")

print(f"\nDataset total: {len(dataset_alpaca)} ejemplos")
print("\nEjemplo del dataset:")
print(json.dumps(dataset_alpaca[0], indent=2, ensure_ascii=False))

## 4. Convertir a ChatML y configurar SFTTrainer con LoRA

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, get_peft_model

def alpaca_a_chatml(ejemplo: dict) -> str:
    """Convierte un ejemplo Alpaca al formato ChatML."""
    instruccion = ejemplo["instruction"]
    if ejemplo.get("input"):
        instruccion += f"\n\nContexto: {ejemplo['input']}"
    return f"""<|im_start|>user
{instruccion}
<|im_end|>
<|im_start|>assistant
{ejemplo['output']}
<|im_end|>"""

# Preparar dataset
dataset_chatml = [{"text": alpaca_a_chatml(ej)} for ej in dataset_alpaca]
dataset_hf = Dataset.from_list(dataset_chatml)

# Cargar modelo base
print("Cargando GPT-2 con LoRA...")
MODELO_BASE = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODELO_BASE)

# Configurar LoRA
config_lora = LoraConfig(
    r=8,              # Rango de las matrices LoRA
    lora_alpha=16,    # Factor de escala
    target_modules=["c_attn", "c_proj"],  # Capas a adaptar en GPT-2
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model_lora = get_peft_model(model, config_lora)

# Mostrar parámetros entrenables
total = sum(p.numel() for p in model_lora.parameters())
entrenables = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"Parámetros totales: {total:,}")
print(f"Parámetros LoRA entrenables: {entrenables:,} ({entrenables/total*100:.2f}%)")

# Configurar SFTTrainer
config_sft = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    max_seq_length=256,
    logging_steps=5,
    save_strategy="no"
)

trainer = SFTTrainer(
    model=model_lora,
    args=config_sft,
    train_dataset=dataset_hf,
    tokenizer=tokenizer,
)

print(f"\nSFTTrainer configurado. Dataset: {len(dataset_hf)} ejemplos en formato ChatML.")

## 5. Chat templates de diferentes modelos

In [ ]:
import pandas as pd

# Tabla de chat templates por modelo
templates = pd.DataFrame([
    {"Modelo": "GPT-4 / ChatGPT", "Formato": "ChatML", "Sistema": "<|im_start|>system", "Usuario": "<|im_start|>user"},
    {"Modelo": "Llama 3.x", "Formato": "Llama 3", "Sistema": "<|begin_of_text|><|start_header_id|>system", "Usuario": "<|start_header_id|>user"},
    {"Modelo": "Mistral 7B", "Formato": "Mistral", "Sistema": "[INST] <<SYS>>", "Usuario": "[INST]"},
    {"Modelo": "Gemma", "Formato": "Gemma", "Sistema": "N/A (no system token)", "Usuario": "<start_of_turn>user"},
    {"Modelo": "Qwen 2.x", "Formato": "ChatML", "Sistema": "<|im_start|>system", "Usuario": "<|im_start|>user"},
])

print("=== CHAT TEMPLATES POR MODELO ===")
print(templates.to_string(index=False))

print("""
=== RESUMEN DEL FLUJO COMPLETO DE INSTRUCTION TUNING ===
1. Recopilar o generar dataset de instrucciones (mínimo 1000-5000 ejemplos)
2. Convertir al formato del modelo base objetivo
3. Aplicar LoRA para reducir parámetros entrenables al 1-5%
4. Entrenar con SFTTrainer (1-3 épocas con lr=2e-4)
5. Evaluar en conjunto de validación (pérdida, perplejidad)
6. Fusionar adaptadores LoRA con el modelo base para despliegue
   → model.merge_and_unload()
""")